# 03 — User Feedback

The previous lessons wired events to Python — clicks collected coordinates, buttons toggled layers. This lesson closes the loop: **the map responds visually to what the user does**.

That shift is what turns a map from a picture into a **tool**.

```
user input  →  coordinates  →  spatial calculation  →  map visualization
```

By the end of this lesson that pipeline will be running end-to-end. It is the same pipeline that drives Point-In-Polygon, Distance, and Bearing — the lessons immediately ahead.

## Setup

In [ ]:
import json, math
from ipyleaflet import Map, Marker, Polyline, GeoJSON
from ipywidgets import HTML, Label, VBox, HBox, Output

WICHITA_FALLS = (33.9137, -98.4934)

with open("../02-Viewing_GeoJSON/data/wichita_falls.geojson") as f:
    geojson = json.load(f)

## Popup Information

A popup attached to a marker is the simplest form of feedback — click a location, see its data. We covered attaching static HTML popups to pre-built markers earlier. Here the popup content is **generated at click time** from the actual coordinates.

In [ ]:
m = Map(center=WICHITA_FALLS, zoom=12)

def on_click(**kwargs):
    if kwargs.get("type") != "click":
        return

    lat, lon = kwargs["coordinates"]

    marker = Marker(location=(lat, lon))
    marker.popup = HTML(value=f"""
        <b>Clicked location</b><br>
        Lat:&nbsp; <code>{lat:.5f}</code><br>
        Lon: <code>{lon:.5f}</code>
    """)
    m.add(marker)

m.on_interaction(on_click)
m

Click the map, then click the marker that appears to see the popup.

## Status Labels with `Output`

Popups are attached to specific features. Sometimes you want a **persistent status area** — a panel next to the map that updates as the user interacts, rather than popups that pile up.

`ipywidgets.Output` is a capture widget: anything you `print()` inside a `with output:` block appears there instead of the default cell output.

In [ ]:
m = Map(center=WICHITA_FALLS, zoom=12)
status = Output(layout={"border": "1px solid #ccc", "padding": "6px", "min_height": "60px"})

def on_click(**kwargs):
    if kwargs.get("type") != "click":
        return

    lat, lon = kwargs["coordinates"]

    status.clear_output(wait=True)
    with status:
        print(f"Last click")
        print(f"  lat: {lat:.5f}")
        print(f"  lon: {lon:.5f}")

m.on_interaction(on_click)
VBox([m, status])

The status panel clears and refreshes with every click — no growing list of markers, no cluttered map.

## Highlighting Selected Features

A GeoJSON layer's style can be changed after it is on the map — update the `style` property and ipyleaflet redraws it instantly. This lets you highlight a feature in response to a click.

The approach: keep one layer per feature, store them in a dict, then on each click replace the style of the previously selected layer and apply a highlight style to the newly selected one.

First, we need click events on the **features themselves** — not the map background. `GeoJSON` layers support an `on_click` parameter for exactly this.

In [ ]:
DEFAULT_STYLE    = {"color": "#2980b9", "fillColor": "#2980b9", "fillOpacity": 0.35, "weight": 2}
HIGHLIGHT_STYLE  = {"color": "#e74c3c", "fillColor": "#e74c3c", "fillOpacity": 0.65, "weight": 3}

m = Map(center=WICHITA_FALLS, zoom=12)
status = Output(layout={"border": "1px solid #ccc", "padding": "6px"})

selected = [None]  # one-element list so the closure can reassign it

# Build one layer per feature so each can be styled independently
feature_layers = []
for feat in geojson["features"]:
    layer = GeoJSON(
        data=feat,
        style=DEFAULT_STYLE.copy()
    )

    def make_handler(lyr, feature):
        def handler(**kwargs):
            # Restore previous selection
            if selected[0] is not None:
                selected[0].style = DEFAULT_STYLE.copy()

            # Highlight the clicked layer
            lyr.style = HIGHLIGHT_STYLE.copy()
            selected[0] = lyr

            status.clear_output(wait=True)
            with status:
                print(f"Selected: {feature['properties']['name']}")
                print(f"  type: {feature['properties']['type']}")
                print(f"  geometry: {feature['geometry']['type']}")
        return handler

    layer.on_click(make_handler(layer, feat))
    feature_layers.append(layer)
    m.add(layer)

VBox([m, status])

Click any feature on the map. It highlights red and the status panel shows its properties. Click a different feature — the previous one resets and the new one highlights.

## Drawing a Line Between Two Clicks

Two clicks → two coordinates → one line. This is the minimal building block for distance and bearing calculations.

The pattern: store the **first click**, wait for the **second click**, then draw the line between them.

In [ ]:
m = Map(center=WICHITA_FALLS, zoom=12)
status = Output(layout={"border": "1px solid #ccc", "padding": "6px"})

points = []  # holds at most two (lat, lon) tuples
line = Polyline(locations=[], color="#e67e22", weight=3)
m.add(line)

def on_click(**kwargs):
    if kwargs.get("type") != "click":
        return

    lat, lon = kwargs["coordinates"]

    if len(points) == 2:
        # Reset: start a fresh pair
        points.clear()
        line.locations = []
        for lyr in list(m.layers):
            if isinstance(lyr, Marker):
                m.remove(lyr)

    points.append((lat, lon))
    m.add(Marker(location=(lat, lon)))

    status.clear_output(wait=True)
    with status:
        if len(points) == 1:
            print(f"Point A: ({lat:.5f}, {lon:.5f})")
            print("Click again to set Point B.")
        else:
            line.locations = points.copy()
            print(f"Point A: ({points[0][0]:.5f}, {points[0][1]:.5f})")
            print(f"Point B: ({points[1][0]:.5f}, {points[1][1]:.5f})")
            print("Click again to start over.")

m.on_interaction(on_click)
VBox([m, status])

Click once to set Point A, click again to set Point B and draw the line. A third click resets and starts over.

## Displaying Computed Information

The line is on the map — but we can go further. Compute something from the two coordinates and display the result alongside the line. Here we compute the straight-line (Euclidean) distance in degrees as a placeholder; the next module will replace this with proper geodesic distance.

The key insight: **the map is an interface for spatial queries**. The user defines the inputs by clicking; your code runs the calculation and feeds the result back.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Approximate great-circle distance in kilometres."""
    R = 6371
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat / 2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon / 2)**2
    return R * 2 * math.asin(math.sqrt(a))

m = Map(center=WICHITA_FALLS, zoom=12)
status = Output(layout={"border": "1px solid #ccc", "padding": "6px"})

points = []
line = Polyline(locations=[], color="#8e44ad", weight=3)
m.add(line)

def on_click(**kwargs):
    if kwargs.get("type") != "click":
        return

    lat, lon = kwargs["coordinates"]

    if len(points) == 2:
        points.clear()
        line.locations = []
        for lyr in list(m.layers):
            if isinstance(lyr, Marker):
                m.remove(lyr)

    points.append((lat, lon))
    m.add(Marker(location=(lat, lon)))

    status.clear_output(wait=True)
    with status:
        if len(points) == 1:
            print(f"Point A: ({lat:.5f}, {lon:.5f})")
            print("Click Point B to measure.")
        else:
            line.locations = points.copy()
            dist = haversine_km(*points[0], *points[1])
            print(f"Point A:  ({points[0][0]:.5f}, {points[0][1]:.5f})")
            print(f"Point B:  ({points[1][0]:.5f}, {points[1][1]:.5f})")
            print(f"Distance: {dist:.2f} km  /  {dist * 0.621371:.2f} mi")
            print("Click to reset.")

m.on_interaction(on_click)
VBox([m, status])

Click two locations. The line draws and the distance appears immediately. This is a complete spatial query tool built in under 30 lines.

The Distance and Bearing lessons will expand this formula and apply it to real data — but the map interaction layer will look exactly like this.

## Exercise A

Extend the two-click line tool from the teaching cells to also place a **midpoint marker** between Point A and Point B.

- Midpoint formula: `((lat1+lat2)/2, (lon1+lon2)/2)`
- Style it differently from the endpoint markers (use an `AwesomeIcon` with a different color or icon)
- Display the distance in the status panel as before

In [ ]:
from ipyleaflet import Map, Marker, Polyline

m = Map(center=(33.9137, -98.4934), zoom=11)
clicks = []
line = None

def handle_click(**kwargs):
    global line
    if kwargs.get("type") != "click":
        return
    clicks.append(kwargs["coordinates"])
    m.add(Marker(location=kwargs["coordinates"]))
    if len(clicks) == 2:
        if line and line in m.layers:
            m.remove_layer(line)
        line = Polyline(locations=clicks, color="red", weight=4)
        m.add(line)
        (lat1, lon1), (lat2, lon2) = clicks
        midpoint = ((lat1 + lat2) / 2, (lon1 + lon2) / 2)
        m.add(Marker(location=midpoint, title="Midpoint"))

m.on_interaction(handle_click)
m


## Exercise B

Build a click counter using a `Label` widget:

- Every click increments the counter and updates the label (e.g. `"Clicks: 4"`)
- Each click also adds a marker to the map
- A **Reset** button zeroes the counter, updates the label, and removes all placed markers

In [ ]:
from ipywidgets import Button, Output

counter = {"value": 0}
status = Output()
reset = Button(description="Reset")


def handle_click(**kwargs):
    if kwargs.get("type") == "click":
        counter["value"] += 1
        with status:
            status.clear_output()
            print(f"Clicks: {counter['value']}")


def reset_count(_):
    counter["value"] = 0
    with status:
        status.clear_output()
        print("Clicks: 0")

m.on_interaction(handle_click)
reset.on_click(reset_count)
reset, status


A strong feedback loop keeps users oriented: every click should update both the map and the status text so they can see what the tool captured. The reset button should clear the running state immediately so the UI never leaves stale counts on screen.

## What This Module Prepared You For

Every lesson in this module was building toward one mental model:

```
User input
     ↓
Coordinates
     ↓
Spatial calculation
     ↓
Map visualization
```

You now have all four stages running. The next modules replace the placeholder `haversine_km` with proper implementations, add bearing calculations, and extend the query from two points to points-within-polygons. The map interaction layer — events, handlers, output widgets — stays exactly the same.

## Next

In [05 — Coordinate Geometry](../05-Coordinate_Geometry/00-Coordinate_Ranges.ipynb), we start doing real math on the coordinates we've been collecting — bounding boxes, ranges, and why lat/lon behaves unexpectedly.